# Module 2.1: Local Network Discovery, IP Allocation & Hostname Resolution

## Homelab NAS Development - Step 2: Home Network Connection

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Discover headless devices on a Local Area Network (LAN) using mDNS and subnet scanning tools.
2. Understand the Multicast DNS (mDNS / Avahi) protocol for `.local` name resolution.
3. Understand DHCP Reservation vs. Static Client IP configuration.
4. Establish a permanent DHCP MAC-reservation on your home router.

## 2.1.1 Network Architecture & Hostname Resolution

A server must possess a predictable IP address. If your Raspberry Pi receives an arbitrary, shifting IP address from the home router's DHCP pool every time it reboots, SSH sessions, Docker volume bindings, and mobile backup syncs will break.

```
+-----------------------------------------------------------+
|                       Home Router                         |
|   Gateway Admin: 10.0.0.1 (Xfinity) or 192.168.1.1        |
|   Subnet: 10.0.0.0/24 (Pool: 10.0.0.2 ~ 10.0.0.253)       |
|       or 192.168.1.0/24 (Pool: 192.168.1.2 ~ .254)        |
+-----------------------------+-----------------------------+
                              |
              +---------------+---------------+
              |                               |
       [Home Devices]                 [Raspberry Pi NAS]
    (Dynamic DHCP Lease)         MAC: 2C:CF:67:xx:xx:xx
                                 Reserved Static IP: 10.0.0.150
```

### Multicast DNS (mDNS) Protocol Anatomy
Multicast DNS (implemented by Avahi on Linux and Bonjour on macOS/Windows) broadcasts UDP packets across port 5353 to the multicast address `224.0.0.251`. Any device on the same local subnet can resolve `homelab-pi.local` without needing a dedicated DNS server. *Note: mDNS broadcasts do not cross across separate VLANs or isolated Guest Wi-Fi networks by default.*

### DHCP Reservation vs. Static Client IP
- **Static IP on Device (`/etc/dhcpcd.conf` or `nmcli`)**: The device statically binds an IP. If your router changes or you connect the device to another network for maintenance, you can suffer IP conflicts or total loss of connectivity.
- **DHCP MAC Reservation (Best Practice)**: The router assigns the identical IP every time based on the Raspberry Pi's hardware MAC address. The Pi remains a standard DHCP client while maintaining a permanent, unchanging IP.

## 2.1.2 Step-by-Step Laboratory Guide

### Task 2.1.1: Local Network Discovery

Verify network connectivity and discover the Raspberry Pi on your Local Area Network (LAN) using multicast DNS (`.local` hostname resolution) or subnet scanning from your workstation. Choose the commands matching your workstation's operating system:

#### 1. Hostname Ping & mDNS Verification (Enforcing IPv4)

By default, modern operating systems prioritize IPv6 on local links (which produces replies from an address starting with `2601:` or `fe80:`). To enforce an IPv4 lookup and discover your Pi's local network IPv4 address (e.g., `10.0.0.xxx` or `192.168.1.xxx`), use the `-4` flag:

**Windows (PowerShell):**
```powershell
# Force IPv4 mDNS resolution (default 4 ICMP echo requests)
ping -4 homelab-pi.local
```

**macOS (Terminal / zsh):**
```bash
# Force IPv4 mDNS resolution (limit to 4 packets)
ping -4 -c 4 homelab-pi.local
```

**Linux (Terminal / bash):**
```bash
# Force IPv4 mDNS resolution (limit to 4 packets)
ping -4 -c 4 homelab-pi.local
```

> **Expected Output:** You should observe replies with low latency (typically 1–10 ms on LAN) and 0% packet loss, displaying the Pi's assigned IPv4 address (e.g., `10.0.0.xxx` on Xfinity or `192.168.1.xxx` on standard routers).

---

#### 2. Subnet Scanning & MAC Address Discovery (Fallback & Inspection)

If mDNS resolution does not respond, or if you need to determine the physical MAC address and IPv4 address of the Raspberry Pi for router DHCP reservation, use your platform's network discovery tools:

**Windows (PowerShell):**
```powershell
# Query local IPv4 neighbor cache filtered by Raspberry Pi Foundation MAC prefixes (OUIs)
Get-NetNeighbor -AddressFamily IPv4 | Where-Object { 
    $_.LinkLayerAddress -like "d8-3a-dd*" -or 
    $_.LinkLayerAddress -like "b8-27-eb*" -or 
    $_.LinkLayerAddress -like "dc-a6-32*" -or 
    $_.LinkLayerAddress -like "2c-cf-67*" 
}

# Alternatively, test DNS name resolution directly:
Resolve-DnsName homelab-pi.local
```

**macOS (Terminal):**
```bash
# Method A: Query the local ARP table filtered by Raspberry Pi OUI prefixes
arp -a | grep -i -E "d8:3a:dd|b8:27:eb|dc:a6:32|2c:cf:67"

# Method B: Native macOS Multicast DNS discovery (Bonjour)
dns-sd -G v4v6 homelab-pi.local
```

**Linux (Terminal):**
```bash
# Method A: Inspect the kernel IPv4 neighbor cache
ip neigh | grep -i -E "d8:3a:dd|b8:27:eb|dc:a6:32|2c:cf:67"

# Method B: Direct mDNS resolution via Avahi
avahi-resolve -n homelab-pi.local
```

> **Router Web Interface Alternative:** You can also log directly into your home router's administrative dashboard (commonly `http://10.0.0.1` for Xfinity or `http://192.168.1.1` for other gateways) and open the **Connected Devices**, **DHCP Client Table**, or **Device Map** to locate `homelab-pi` alongside its assigned IP and MAC address.

### 🖼️ Task 2.1.1 Verification: Local Network Discovery & Initial Ping Test

**Verification Objective:** Attach a terminal screenshot from your workstation (PowerShell, macOS Terminal, or Linux shell) demonstrating a successful `ping -4 homelab-pi.local` or network discovery command showing active IPv4 replies and 0% packet loss.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 2.1.2: Configure Router DHCP Reservation

1. Locate the physical MAC address corresponding to your Raspberry Pi's active interface (`eth0` for Ethernet or `wlan0` for Wi-Fi) from the discovery step above.
2. Open your web browser and navigate to your home router administration portal:
   - **Xfinity Gateways:** `http://10.0.0.1` (Default username: `admin`, password: Wi-Fi password or `password`).
   - **Standard Gateways:** `http://192.168.1.1` or `http://192.168.0.1`.
3. Navigate to **Connected Devices** → **Devices** (or **LAN Setup** / **DHCP Server** / **Address Reservation**).
4. Add a Static DHCP Lease / Reserved IP:
   - Choose a static IP within your router's valid DHCP range (e.g., on Xfinity where the valid range is `10.0.0.2` ~ `10.0.0.253`, choose an address like `10.0.0.150` or `10.0.0.200`).
   - Bind this IP permanently to your Raspberry Pi's MAC address.
5. Save and apply the settings.

---

### 📝 Assigned Static IP & Network Record Sheet

*Record your router gateway and the reserved static IP configuration below for reference across future modules:*

> ### 📌 Network Configuration Record
> - **Router Gateway Web Admin:** `http://10.0.0.1` *(or http://192.168.1.1)*
> - **DHCP Subnet Range:** `10.0.0.2 ~ 10.0.0.253` *(or your router's valid DHCP pool)*
> - **Raspberry Pi Physical MAC Address:** `[Enter MAC, e.g. 2C:CF:67:xx:xx:xx]`
> - **Assigned Reserved Static IP:** `[Enter reserved IP, e.g. 10.0.0.200]`

### 🖼️ Task 2.1.2 Verification: Router DHCP Reservation Table

**Verification Objective:** Attach a screenshot from your router administration portal (e.g. Xfinity `http://10.0.0.1` Connected Devices page) showing the assigned static IP reservation mapped to your Raspberry Pi's MAC address.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 2.1.3: Static IP Verification & Secondary Ping Test

Once the router reservation is saved, re-establish the network lease on the Raspberry Pi so it adopts the new reserved IP:
- Either reboot the Pi gracefully via terminal (`sudo reboot`), or
- Unplug and reconnect the Ethernet cable (or toggle the network interface).

Wait ~30–45 seconds for the Pi to reboot and obtain its new IP address from the router. Now, perform a second ping test from your workstation targeting the **newly reserved static IPv4 address** directly as well as the `.local` hostname enforcing IPv4 (`ping -4`):

#### 1. Ping the Reserved Static IPv4 Address Directly

**Windows (PowerShell):**
```powershell
# Ping the newly assigned static IPv4 address directly (e.g., 10.0.0.200)
ping 10.0.0.200
```

**macOS (Terminal / zsh):**
```bash
# Ping the newly assigned static IPv4 address directly (limit to 4 packets)
ping -c 4 10.0.0.200
```

**Linux (Terminal / bash):**
```bash
# Ping the newly assigned static IPv4 address directly (limit to 4 packets)
ping -c 4 10.0.0.200
```

---

#### 2. Verify mDNS Hostname Resolution to the New Static Lease (Enforcing IPv4)

**Windows (PowerShell):**
```powershell
# Verify homelab-pi.local resolves to the new static IPv4 lease
ping -4 homelab-pi.local
```

**macOS (Terminal / zsh):**
```bash
# Verify homelab-pi.local resolves to the new static IPv4 lease
ping -4 -c 4 homelab-pi.local
```

**Linux (Terminal / bash):**
```bash
# Verify homelab-pi.local resolves to the new static IPv4 lease
ping -4 -c 4 homelab-pi.local
```

> **Expected Output:** Both ping tests should return replies with 0% packet loss and low latency (<10 ms). The direct IPv4 ping confirms that your router has permanently bound the static lease to the Pi's MAC address, and the `ping -4` command confirms mDNS resolves directly to the new static IPv4 address.

### 🖼️ Task 2.1.3 Verification: Secondary Ping Test (New Static IP)

**Verification Objective:** Attach a terminal screenshot from your workstation (PowerShell, macOS Terminal, or Linux shell) demonstrating a successful second ping test directly to the newly assigned static IP (e.g., `ping 10.0.0.150` or `ping -4 homelab-pi.local`) with 0% packet loss.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Initial workstation ping to `homelab-pi.local` (`ping -4`) succeeded with 0% packet loss.
- [ ] Raspberry Pi physical MAC address identified via discovery tools or router device list.
- [ ] Router DHCP static lease assigned and recorded in the Network Configuration Record.
- [ ] Router DHCP reservation table screenshot captured and attached.
- [ ] Secondary ping test directly to new reserved static IP verified with 0% loss and screenshot attached.

Proceed to the next module in sequence once this checkpoint is satisfied.